<a href="https://colab.research.google.com/github/boba1987/advanced-neural-networks-and-deep-learning/blob/main/Customer_Support_Ticket_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Customer Support Ticket Classification

Predict **ticket type** (`Incident`, `Request`, `Problem`, `Change`) from ticket **`body` text only** using **TF-IDF + PyTorch MLP** (English only).

**Dataset:** `dataset-tickets-en.csv` (~11,923 rows) | **Split:** 80% train / 20% test


## Colab notes

- **Runtime → Run all** (GPU optional; CPU is fine for this model).
- Data is fetched with **`curl`** from GitHub.


## Phase 1: Setup and data loading


In [ ]:
!pip install -q torch scikit-learn pandas matplotlib seaborn joblib


In [ ]:
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### Load dataset

Input = **`body` only** (no subject, no queue); label = **type**. English rows only (`language == en`).


In [ ]:
GITHUB_RAW_URL = "https://raw.githubusercontent.com/boba1987/advanced-neural-networks-and-deep-learning/main/dataset-tickets-en.csv"
CSV_PATH = "dataset-tickets-en.csv"

!curl -L -o {CSV_PATH} "{GITHUB_RAW_URL}"

raw = pd.read_csv(CSV_PATH)
raw["body"] = raw["body"].astype(str).str.strip()
raw = raw[raw["language"].astype(str).str.lower() == "en"].copy()

df = pd.DataFrame({
    "ticket_text": raw["body"],
    "category": raw["type"],
}).dropna()

df = df[df["ticket_text"].str.len() > 0].reset_index(drop=True)

print(f"Loaded {len(df):,} tickets")
print(f"Categories: {sorted(df['category'].unique())}")
print(df["category"].value_counts())


### Exploratory analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["category"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Tickets per type")
axes[0].tick_params(axis="x", rotation=45)
df["ticket_text"].str.len().hist(bins=50, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Body length")
plt.tight_layout()
plt.show()


## Phase 2: Preprocessing, labels, and split


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["text_clean"] = df["ticket_text"].apply(clean_text)

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["category"])
num_classes = len(label_encoder.classes_)
id_to_label = {i: label_encoder.classes_[i] for i in range(num_classes)}
print("Label mapping:", id_to_label)


In [ ]:
texts_raw = df["ticket_text"].values
texts_mlp = df["text_clean"].values
labels = df["label"].values

X_train_full, X_test_orig, y_train_full, y_test, texts_mlp_train_full, texts_mlp_test = train_test_split(
    texts_raw, labels, texts_mlp,
    test_size=0.2, random_state=42, stratify=labels,
)

X_train, X_val, y_train, y_val, X_mlp_train, X_mlp_val = train_test_split(
    X_train_full, y_train_full, texts_mlp_train_full,
    test_size=0.1, random_state=42, stratify=y_train_full,
)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test_orig):,}")


## Phase 3: TF-IDF + MLP


In [ ]:
HIDDEN_SIZES = (256, 128, 64)
DROPOUTS = (0.5, 0.4, 0.3)
MAX_FEATURES = 5000
BATCH_SIZE = 64
MAX_EPOCHS = 15
EARLY_STOP_PATIENCE = 3

class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

text_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES, ngram_range=(1, 3), min_df=2, max_df=0.95, sublinear_tf=True,
)

X_train_sparse = text_vectorizer.fit_transform(X_mlp_train)
X_val_sparse = text_vectorizer.transform(X_mlp_val)
X_test_sparse = text_vectorizer.transform(texts_mlp_test)
input_size = X_train_sparse.shape[1]
print(f"TF-IDF features: {input_size:,}")

class SparseDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse.tocsr()
        self.y = y.astype(np.int64)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        row = self.X[idx].toarray().astype(np.float32).squeeze()
        return torch.from_numpy(row), torch.tensor(self.y[idx], dtype=torch.long)

class TicketClassifier(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes=HIDDEN_SIZES, dropouts=DROPOUTS):
        super().__init__()
        layers, in_dim = [], input_size
        for h, d in zip(hidden_sizes, dropouts):
            layers.extend([nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(d)])
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

pin_memory = device.type == "cuda"
train_loader = DataLoader(SparseDataset(X_train_sparse, y_train), batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
val_loader = DataLoader(SparseDataset(X_val_sparse, y_val), batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)
test_loader = DataLoader(SparseDataset(X_test_sparse, y_test), batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

model = TicketClassifier(input_size, num_classes).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = 0
epochs_no_improve = 0
best_state = None

print(f"MLP parameters: {sum(p.numel() for p in model.parameters()):,}")
